In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
import warnings

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('train.csv')
df.sample(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
104,105,0,3,"Gustafsson, Mr. Anders Vilhelm",male,37.0,2,0,3101276,7.925,NaN,S
806,807,0,1,"Andrews, Mr. Thomas Jr",male,39.0,0,0,112050,0.000,A36,S


## Plan the pipeline first

In [3]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'], inplace=True)

In [4]:
# step1 -> tts
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['Survived']),
                                                    df['Survived'], 
                                                    test_size=0.2,
                                                    random_state=42)

In [5]:
X_train.head(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5,S
733,2,male,23.0,0,0,13.0,S


In [6]:
y_train.head(2)

331    0
733    0
Name: Survived, dtype: int64

## create all transformers

In [7]:
# 1st transformer
# imputation transformer
trf1 = ColumnTransformer([
    ('impute_age', SimpleImputer(), [2]),
    ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6])
],remainder='passthrough')

In [8]:
# 2nd transformer
# one hot encoding
trf2 = ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(sparse=False, handle_unknown='ignore'), [1,6])
], remainder='passthrough')

In [9]:
# 3rd transformer
# Scaling
trf3 = ColumnTransformer([
    ('scale', MinMaxScaler(), slice(0,10))
])

In [10]:
# 4th transformer
# feature selection
trf4 = SelectKBest(score_func=chi2, k=5)

In [11]:
# 5th transformer
# train the model
trf5 = DecisionTreeClassifier()

## creating pipeline

In [12]:
pipe = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2),
    ('trf3', trf3),
    ('trf4', trf4),
    ('trf5', trf5),
])

## Pipeline VS make_pipeline

### Pipeline requires naming of steps, make_pipeline does not.
### Same applies to ColumnTransformer vs make_column_transformer

In [13]:
# alternate syntax
# pipe = make_pipeline(trf1, trf2, trf3, trf4, trf5)

In [14]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=5,
                             score_func=<function chi2 at 0x7bae6db19da0>)),
                ('trf5', DecisionTreeClassifier())])

## Exploring Pipeline

In [15]:
pipe.named_steps['trf1'].transformers_[0][1].statistics_

array([29.49884615])

In [16]:
pipe.named_steps['trf1'].transformers_[1][1].statistics_

array(['S'], dtype=object)

In [17]:
# Display pipeline
from sklearn import set_config
set_config(display='diagram')

In [18]:
y_pred = pipe.predict(X_test)

In [19]:
from sklearn.metrics import accuracy_score
accuracy_score(y_pred, y_test)

0.6256983240223464

In [20]:
display(pipe)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=5,
                             score_func=<function chi2 at 0x7bae6db19da0>)),
                ('trf5', DecisionTreeClassifier())])

## CrossValidation Using pipeline

In [21]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train, cv = 5, scoring='accuracy').mean()

0.6391214419383433

## GridSearch using Pipeline

In [22]:
# gridsearchcv
params = {
    'trf5__max_depth' : [1,2,3,4,5,None]
}

In [23]:
from sklearn.model_selection import GridSearchCV

In [24]:
grid = GridSearchCV(pipe, params, cv = 5, scoring='accuracy')
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('trf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('trf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse=False),
                                                                         [1,
                                                                          6])])),
                                       ('trf3',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('trf4',
                                        SelectKBest(k=5,
                                                    score_func=<function chi2 at 0x7bae6db19da0>)),
                                       ('trf5', DecisionTreeClassifier())]),
             param_grid={'trf5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [25]:
grid.best_score_

0.6391214419383433

In [26]:
grid.best_params_

{'trf5__max_depth': 2}

## Exporting the pipeline

In [27]:
import pickle
pickle.dump(pipe, open('pipe.pkl', 'wb'))